# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [2]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [3]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [4]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

STEMS = {
    "drums": "drums.wav",
    "vocals": "vocals.wav",
    "bass": "bass.wav",
    "other": "other.wav"
}

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

GENRE_TO_TEST = 'rock'

SONG_INDEX = 0 


**Complete the function `build_dataset` in question notebook and answer following questions (Q1 to Q3).Hint: 1kb = 1024 bytes**


In [5]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, "genres_stems", genre)
        if not os.path.isdir(genre_path):
            continue

        valid_songs = []

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            complete = True
            for stem_file in STEMS.values():
                stem_path = os.path.join(song_path, stem_file)
                if not os.path.exists(stem_path):
                    complete = False
                    break

            if complete:
                valid_songs.append(song_path)

        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        for song_path in train_songs:
            for key, stem_file in STEMS.items():
                train_dataset[genre][key].append(os.path.join(song_path, stem_file))

        for song_path in val_songs:
            for key, stem_file in STEMS.items():
                val_dataset[genre][key].append(os.path.join(song_path, stem_file))

        # Helper function to populate dict
        def add_to_dict(target_dict, song_list):
            pass

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

**What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)**

In [6]:
corrupted = 0
small_mb = 0

for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, "genres_stems", genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue
        for stem in STEMS.values():
            f = os.path.join(song_path, stem)
            if os.path.exists(f):
                size = os.path.getsize(f)
                if size < 4096:
                    corrupted += 1
                if size < 5.0491 * 1024 * 1024:
                    small_mb += 1

print(corrupted + small_mb)

1256


**What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ?**

In [7]:
big = 0
small = 0

for genre in GENRES:
    genre_path = os.path.join(DATA_ROOT, "genres_stems", genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        if not os.path.isdir(song_path):
            continue
        for stem in STEMS.values():
            f = os.path.join(song_path, stem)
            if os.path.exists(f):
                size = os.path.getsize(f)
                if size > 5.0493 * 1024 * 1024:
                    big += 1
                if size < 5.0491 * 1024 * 1024:
                    small += 1

print(abs(big - small))

1072


**What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?**

In [8]:
print(abs(
    len(tr["reggae"]["drums"]) -
    len(val["country"]["vocals"])
))

66


**Complete the function `find_long_silences` in question notebook and answer following questions (Q4 to Q9)**

In [9]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                try:
                    y, sr = librosa.load(file_path, sr=sr)
                    total_duration = len(y) / sr

                    intervals = librosa.effects.split(y, top_db=top_db)

                    silence_type = []
                    max_silence = 0

                    if len(intervals) == 0:
                        max_silence = total_duration
                        silence_type.append("FULL")

                    else:
                        # START silence
                        if intervals[0][0] > 0:
                            start_sil = intervals[0][0] / sr
                            max_silence = max(max_silence, start_sil)
                            silence_type.append("START")

                        # END silence
                        if intervals[-1][1] < len(y):
                            end_sil = (len(y) - intervals[-1][1]) / sr
                            max_silence = max(max_silence, end_sil)
                            silence_type.append("END")

                        # MIDDLE silence
                        for i in range(1, len(intervals)):
                            gap = (intervals[i][0] - intervals[i-1][1]) / sr
                            if gap > 0:
                                max_silence = max(max_silence, gap)
                                silence_type.append("MIDDLE")

                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(set(silence_type)),
                            "File_Path": file_path
                        })

                except:
                    pass

    return pd.DataFrame(records)
# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

pivot = df_silence.pivot_table(index="Genre",
                               columns="Stem",
                               aggfunc="size",
                               fill_value=0)

print(pivot)


Stem       bass  drums  other  vocals
Genre                                
blues        17     25      6      45
classical    66     57      4      70
country      11     14      1      17
disco         7      2      3      18
hiphop       20      3     17       4
jazz         26     20      1      73
metal         6      2      1      42
pop          10      4      2       5
reggae        6      4      7      14
rock         12      8      1      27


In [10]:
df_silence = find_long_silences(tr, threshold_sec=5, top_db=TOP_DB)

**Total number of sound files having silence greater than equal to 5 secs**

In [11]:
print(len(df_silence))

678


**Total number of sound tracks in Vocals where silence >= 5 secs**

In [12]:
print(len(df_silence[df_silence["Stem"] == "vocals"]))

315


**What's the average Silence Length in Vocals (in secs)**

In [13]:
print(round(
    df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean(), 2
))

12.78


**Total number of drums sound tracks in jazz where silence >= 5 secs**

In [14]:
print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
]))

20


**Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle**

In [15]:
print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "MIDDLE")
]))

0


**Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10**

In [16]:
print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
]))

7


**Select the first song from the ‘rock’ genre, combine all stems to prepare a sample, perform the tasks outlined in the notebook, and answer questions 10–12 based on the results.**

In [17]:
stems_audio = []
try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, sr = librosa.load(file_path, sr=SR, duration=5.0)
        stems_audio.append((key, y, sr))

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

Audio loaded successfully.


In [18]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.array([y for _, y, _ in stems_audio])

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))

# Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------

**What is the length of the mix sample?**

In [19]:
print(len(mix_raw))

110250


**What is the value of RMS Amplitude of mix sample?**

In [20]:
print(round(rms_val, 2))

0.17


**What is the value of max value of peak  normalized sample ?**

In [21]:
print(round(np.max(np.abs(mix_norm)), 2))

1.0
